In [ ]:
pip install langchain-pinecone pinecone-client

In [ ]:
pip install sentence-transformers

In [ ]:
from dotenv import load_dotenv
import os
load_dotenv()

True

<b>Load PDF</b>

In [5]:
from langchain_community.document_loaders import PyPDFLoader

loader=PyPDFLoader("D:\\obssessed\\PDFS\\Gen-Ai\\7_document_Loader\\O2h_resume.pdf")

docs=loader.load()

<b>Split Documents</b>

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter=RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks=splitter.split_documents(docs)

print(docs)

[Document(metadata={'producer': 'WeasyPrint 62.3', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Resume - Harshil Solanki', 'source': 'D:\\obssessed\\PDFS\\Gen-Ai\\7_document_Loader\\O2h_resume.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}, page_content='HARSHIL SOLANKIFull-Stack Developer | AI/ML Integration\n+91 7016978473  |  harshilsolanki2610@gmail.com  |  LinkedIn  |  GitHub\nPROFESSIONAL SUMMARY\nPassionate and growth-oriented Full-Stack Developer eager to kick-start a career contributing to scalable software solutions.\nExperienced managing the entire software development life cycle—from initial conceptualization to final web deployment. Highly\ncomfortable  balancing  front-end  UI  design  with  robust  back-end  tech  stacks,  maintaining  an  eye  for  functionality,  and\ndemonstrating a strong enthusiasm for learning. \nTECHNICAL SKILLS\nLanguages: JavaScript, TypeScript, Python, Java, Dart\nFrontend: React.js, Next.js, Flutter, Tailwind CSS, Shadcn/UI\nBacken

<B>Embeddings</B>

In [9]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6661.73it/s]


In [32]:
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore

pc=Pinecone(
    api_key=os.getenv('PINECONE_API_KEY')
)
index_name='chatpdf'

vectorstore = PineconeVectorStore.from_documents(
    documents=chunks,
    embedding=embeddings,
    index_name=index_name
)



<B>Retrievr</B>

In [48]:
retriever=vectorstore.as_retriever(
    search_kwargs={'k':4},
    
)

<B>LLM</B>

In [35]:

import os
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(
    model="openai/gpt-oss-120b:free",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
)

<B>Prompt</B>

In [36]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
Answer the question using only the provided context.

Context:
{context}

Question:
{question}

If the answer is not in the context, say:
"I don't know."
""")

<B>Output Parser</B>

In [37]:
from langchain_core.output_parsers import StrOutputParser


parser=StrOutputParser()

In [41]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [49]:
from langchain_core.runnables import RunnablePassthrough

chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | parser
)

In [50]:
response = chain.invoke(
    "What is technical skill list out all of them?"
)

print(response)

**Technical Skills**

- **Languages:** JavaScript, TypeScript, Python, Java, Dart  
- **Frontend:** React.js, Next.js, Flutter, Tailwind CSS, Shadcn/UI  
- **Backend & API:** Node.js, Express.js, NestJS, tRPC  
- **Databases:** MongoDB, PostgreSQL (Neon), MS SQL, SQLite, Prisma ORM  
- **Tools & Authentication:** Polar Payments, Better Auth, Clerk Auth, Stream Chat/Video, JWT, Vercel, Git


In [45]:
chain.get_graph().print_ascii()

           +---------------------------------+        
           | Parallel<context,question>Input |        
           +---------------------------------+        
                    **              ***               
                 ***                   **             
               **                        ***          
+----------------------+                    **        
| VectorStoreRetriever |                     *        
+----------------------+                     *        
            *                                *        
            *                                *        
            *                                *        
       +--------+                     +-------------+ 
       | format |                     | Passthrough | 
       +--------+***                  +-------------+ 
                    **              **                
                      ***        ***                  
                         **    **                     
          